In [1]:
pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 55.0 MB/s eta 0:00:00


In [2]:
test_label = "/content/runs/predict_instrument_dataset_16/labels/frame225.txt"

print("🔍 ANALISI FILE GT")
print("="*70)
def extract_bbox_from_polygon_label(label_file):
    """
    Legge file YOLO segmentation e estrae bbox da poligoni

    Args:
        label_file: Path al file .txt con formato: class x1 y1 x2 y2 ... xN yN

    Returns:
        classes: Lista di class_id
        boxes: Lista di bbox in formato YOLO [x_center, y_center, w, h]
    """
    classes = []
    boxes = []

    if not os.path.exists(label_file):
        return classes, boxes

    with open(label_file, 'r') as f:
        for line in f:
            parts = line.strip().split()

            if len(parts) < 5:  # Minimo: class + 2 punti (x1 y1 x2 y2)
                continue

            class_id = int(float(parts[0]))

            # Estrai coordinate poligono (skip class_id)
            coords = [float(x) for x in parts[1:]]

            if len(coords) < 4 or len(coords) % 2 != 0:
                continue

            # Separa x e y
            x_coords = coords[0::2]  # Indici pari: x
            y_coords = coords[1::2]  # Indici dispari: y

            # Calcola bounding box
            x_min = min(x_coords)
            x_max = max(x_coords)
            y_min = min(y_coords)
            y_max = max(y_coords)

            # Converti in formato YOLO
            x_center = (x_min + x_max) / 2
            y_center = (y_min + y_max) / 2
            width = x_max - x_min
            height = y_max - y_min

            classes.append(class_id)
            boxes.append([x_center, y_center, width, height])

    return classes, boxes

classes, boxes = extract_bbox_from_polygon_label(test_label)

print(f"File: {test_label}")
print(f"Numero oggetti: {len(classes)}")
print(f"\nDettagli:")

for i, (cls, bbox) in enumerate(zip(classes, boxes)):
    x_c, y_c, w, h = bbox
    print(f"  {i+1}. Classe {cls} ({CLASS_NAMES[cls]})")
    print(f"     Bbox: x={x_c:.3f}, y={y_c:.3f}, w={w:.3f}, h={h:.3f}")
    print(f"     Area: {w*h:.4f} ({w*h*100:.2f}% dell'immagine)")

🔍 ANALISI FILE GT


NameError: name 'os' is not defined

In [3]:
import os
import numpy as np
from pathlib import Path
from ultralytics import YOLO
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import cv2
from collections import Counter, defaultdict
import time

# ============================================================================
# NUOVA FUNZIONE: OVERLAY GT BOUNDING BOXES SU IMMAGINI
# ============================================================================

def overlay_gt_boxes_on_predictions(pred_folder, image_folder, all_gt_data, class_names):
    """
    Sovrappone le GT bounding boxes alle immagini predette da YOLO

    Args:
        pred_folder: Cartella con le predizioni YOLO (contiene sottocartella con immagini)
        image_folder: Cartella con immagini originali
        all_gt_data: Lista di dict con 'image', 'classes', 'boxes' GT
        class_names: Nomi delle classi
    """
    print(f"\n  🎨 Sovrapposizione GT boxes alle predizioni...")

    # Trova la cartella con le immagini predette
    pred_images_folder = pred_folder  # YOLO salva in pred_folder direttamente

    if not os.path.exists(pred_images_folder):
        print(f"  ⚠️ Cartella predizioni non trovata: {pred_images_folder}")
        return

    # Crea cartella output per immagini con GT
    output_folder = os.path.join(pred_folder, "with_gt_boxes")
    os.makedirs(output_folder, exist_ok=True)

    # Colori GT (diversi da quelli YOLO) - Verde acceso per GT
    gt_color = (0, 255, 0)  # Verde BGR
    gt_thickness = 3

    processed = 0

    for gt_data in all_gt_data:
        img_name = gt_data['image']
        gt_classes = gt_data['classes']
        gt_boxes = gt_data.get('boxes', [])

        if not gt_boxes:
            continue

        # Trova immagine predetta (YOLO può salvare come .jpg)
        pred_img_path = None
        for ext in ['.jpg', '.png', '.jpeg']:
            potential_path = os.path.join(pred_images_folder, f"{img_name}{ext}")
            if os.path.exists(potential_path):
                pred_img_path = potential_path
                break

        if pred_img_path is None:
            continue

        # Leggi immagine predetta
        img = cv2.imread(pred_img_path)
        if img is None:
            continue

        h, w = img.shape[:2]

        # Disegna GT boxes
        for box, cls in zip(gt_boxes, gt_classes):
            if cls < 0 or cls >= len(class_names):
                continue

            # Converti YOLO normalizzato -> pixel
            x_center, y_center, box_w, box_h = box
            x1 = int((x_center - box_w/2) * w)
            y1 = int((y_center - box_h/2) * h)
            x2 = int((x_center + box_w/2) * w)
            y2 = int((y_center + box_h/2) * h)

            # Disegna bbox GT (verde)
            cv2.rectangle(img, (x1, y1), (x2, y2), gt_color, gt_thickness)

            # Etichetta GT
            label = f"GT: {class_names[cls]}"

            # Background per testo
            (text_w, text_h), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)
            cv2.rectangle(img, (x1, y1 - text_h - 10), (x1 + text_w, y1), gt_color, -1)

            # Testo
            cv2.putText(img, label, (x1, y1 - 5),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 2)

        # Salva immagine con GT overlay
        output_path = os.path.join(output_folder, f"{img_name}.jpg")
        cv2.imwrite(output_path, img)
        processed += 1

    print(f"  ✓ Sovrapposte GT boxes su {processed} immagini")
    print(f"  📂 Salvate in: {output_folder}")


# ============================================================================
# FUNZIONI DI MATCHING GT-PRED (mantieni tutto uguale)
# ============================================================================

def iou(box1, box2):
    """Calcola IoU tra due box YOLO normalizzate (x_center, y_center, w, h)."""
    x1_min = box1[0] - box1[2] / 2
    x1_max = box1[0] + box1[2] / 2
    y1_min = box1[1] - box1[3] / 2
    y1_max = box1[1] + box1[3] / 2

    x2_min = box2[0] - box2[2] / 2
    x2_max = box2[0] + box2[2] / 2
    y2_min = box2[1] - box2[3] / 2
    y2_max = box2[1] + box2[3] / 2

    inter_xmin = max(x1_min, x2_min)
    inter_ymin = max(y1_min, y2_min)
    inter_xmax = min(x1_max, x2_max)
    inter_ymax = min(y1_max, y2_max)

    inter_area = max(0, inter_xmax - inter_xmin) * max(0, inter_ymax - inter_ymin)
    box1_area = (x1_max - x1_min) * (y1_max - y1_min)
    box2_area = (x2_max - x2_min) * (y2_max - y2_min)

    union_area = box1_area + box2_area - inter_area
    return inter_area / union_area if union_area > 0 else 0


def match_predictions_to_gt(gt_classes, pred_classes, pred_boxes, gt_boxes, iou_threshold=0.5):
    """
    Matcha predizioni con ground truth basandosi su IoU delle bounding box.
    """
    matched_pairs = []
    gt_matched = [False] * len(gt_classes)
    pred_matched = [False] * len(pred_classes)

    if len(gt_boxes) == 0 or len(pred_boxes) == 0:
        unmatched_gt = list(gt_classes)
        unmatched_pred = list(pred_classes)
        return matched_pairs, unmatched_gt, unmatched_pred

    iou_matrix = np.zeros((len(gt_boxes), len(pred_boxes)))
    for i, gt_box in enumerate(gt_boxes):
        for j, pred_box in enumerate(pred_boxes):
            iou_val = iou(gt_box, pred_box)
            iou_matrix[i, j] = iou_val

    while True:
        if np.all(iou_matrix < 0):
            break

        max_iou = np.max(iou_matrix)
        if max_iou < iou_threshold:
            break

        i, j = np.unravel_index(np.argmax(iou_matrix), iou_matrix.shape)

        if not gt_matched[i] and not pred_matched[j]:
            matched_pairs.append((gt_classes[i], pred_classes[j]))
            gt_matched[i] = True
            pred_matched[j] = True
            iou_matrix[i, :] = -1
            iou_matrix[:, j] = -1

    unmatched_gt = [gt_classes[i] for i in range(len(gt_classes)) if not gt_matched[i]]
    unmatched_pred = [pred_classes[j] for j in range(len(pred_classes)) if not pred_matched[j]]

    return matched_pairs, unmatched_gt, unmatched_pred


def match_by_class_only(gt_classes, pred_classes):
    """Matching semplificato per classe"""
    matched_pairs = []
    gt_counter = Counter(gt_classes)
    pred_counter = Counter(pred_classes)
    all_classes = set(gt_counter.keys()) | set(pred_counter.keys())

    for cls in all_classes:
        gt_count = gt_counter.get(cls, 0)
        pred_count = pred_counter.get(cls, 0)
        matches = min(gt_count, pred_count)
        matched_pairs.extend([(cls, cls)] * matches)

        if gt_count > pred_count:
            matched_pairs.extend([(cls, -1)] * (gt_count - pred_count))
        if pred_count > gt_count:
            matched_pairs.extend([(-1, cls)] * (pred_count - gt_count))

    return matched_pairs


# ============================================================================
# STATISTICHE (mantieni tutto uguale)
# ============================================================================

def compute_per_class_statistics(y_true, y_pred, class_names):
    """Calcola statistiche dettagliate per ogni strumento/classe"""
    unique_classes = sorted(set([c for c in y_true if c >= 0] + [c for c in y_pred if c >= 0]))
    per_class_stats = {}

    for class_id in unique_classes:
        if class_id >= len(class_names):
            continue

        class_name = class_names[class_id]
        tp = sum(1 for gt, pred in zip(y_true, y_pred) if gt == class_id and pred == class_id)
        fp = sum(1 for gt, pred in zip(y_true, y_pred) if gt != class_id and pred == class_id)
        fn = sum(1 for gt, pred in zip(y_true, y_pred) if gt == class_id and pred != class_id)
        tn = sum(1 for gt, pred in zip(y_true, y_pred) if gt != class_id and pred != class_id)
        total_gt = sum(1 for gt in y_true if gt == class_id)
        total_pred = sum(1 for pred in y_pred if pred == class_id)

        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        sensitivity = recall
        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0

        y_true_binary = [1 if gt == class_id else 0 for gt in y_true]
        y_pred_binary = [1 if pred == class_id else 0 for pred in y_pred]
        class_accuracy = accuracy_score(y_true_binary, y_pred_binary)

        per_class_stats[class_id] = {
            'name': class_name,
            'precision': precision,
            'recall': recall,
            'sensitivity': sensitivity,
            'specificity': specificity,
            'f1': f1,
            'accuracy': class_accuracy,
            'tp': tp,
            'fp': fp,
            'fn': fn,
            'tn': tn,
            'total_gt': total_gt,
            'total_pred': total_pred
        }

    return per_class_stats

def print_per_class_report(per_class_stats, overall_accuracy):
    """Stampa report dettagliato per ogni strumento"""
    print("\n" + "="*80)
    print("📊 METRICHE DETTAGLIATE PER STRUMENTO")
    print("="*80)
    print(f"🎯 OVERALL ACCURACY: {overall_accuracy:.4f} ({overall_accuracy*100:.2f}%)")
    print("="*80)

    for class_id in sorted(per_class_stats.keys()):
        stats = per_class_stats[class_id]
        print(f"\n{stats['name']} (ID: {class_id})")
        print("-" * 80)
        print(f"  Precision:     {stats['precision']:.4f}  (TP / (TP + FP))")
        print(f"  Recall:        {stats['recall']:.4f}  (TP / (TP + FN))")
        print(f"  Sensitivity:   {stats['sensitivity']:.4f}  (= Recall)")
        print(f"  Specificity:   {stats['specificity']:.4f}  (TN / (TN + FP))")
        print(f"  F1-Score:      {stats['f1']:.4f}")
        print(f"  Accuracy:      {stats['accuracy']:.4f}  ((TP + TN) / Total)")
        print(f"  GT Instances:  {stats['total_gt']}")
        print(f"  Pred Instances:{stats['total_pred']}")
        print(f"  TP: {stats['tp']:4d}  |  FP: {stats['fp']:4d}  |  "
              f"FN: {stats['fn']:4d}  |  TN: {stats['tn']:4d}")

    print("\n" + "="*80)
    print("MACRO AVERAGES")
    print("="*80)

    avg_precision = np.mean([s['precision'] for s in per_class_stats.values()])
    avg_recall = np.mean([s['recall'] for s in per_class_stats.values()])
    avg_f1 = np.mean([s['f1'] for s in per_class_stats.values()])
    avg_sensitivity = np.mean([s['sensitivity'] for s in per_class_stats.values()])
    avg_specificity = np.mean([s['specificity'] for s in per_class_stats.values()])

    print(f"Precision:   {avg_precision:.4f}")
    print(f"Recall:      {avg_recall:.4f}")
    print(f"Sensitivity: {avg_sensitivity:.4f}")
    print(f"Specificity: {avg_specificity:.4f}")
    print(f"F1-Score:    {avg_f1:.4f}")
    print("="*80)


def analyze_instrument_distribution(all_gt_data, all_predictions, class_names):
    """Analizza la distribuzione degli strumenti nel dataset"""
    print("\n" + "="*80)
    print("🔍 DISTRIBUZIONE STRUMENTI NEL DATASET")
    print("="*80)

    gt_counter = Counter()
    pred_counter = Counter()

    for gt_data in all_gt_data:
        gt_counter.update(gt_data['classes'])

    for pred_data in all_predictions:
        pred_counter.update(pred_data['classes'])

    print("\n{:<30} {:>12} {:>12}".format("Strumento", "GT Count", "Pred Count"))
    print("-" * 80)

    all_classes = sorted(set(gt_counter.keys()) | set(pred_counter.keys()))

    for cls in all_classes:
        if cls >= 0 and cls < len(class_names):
            name = class_names[cls]
            gt_count = gt_counter.get(cls, 0)
            pred_count = pred_counter.get(cls, 0)
            print(f"{name:<30} {gt_count:>12} {pred_count:>12}")

    print("-" * 80)
    print(f"{'TOTALE':<30} {sum(gt_counter.values()):>12} {sum(pred_counter.values()):>12}")
    print("="*80)


def compute_metrics_with_matching(all_gt_data, all_predictions, use_bbox_matching=True, iou_threshold=0.5):
    """Calcola metriche matchando GT e predizioni"""
    y_true = []
    y_pred = []

    stats = {
        'matched': 0,
        'false_negatives': 0,
        'false_positives': 0,
        'total_gt': 0,
        'total_pred': 0,
        'per_class_matches': defaultdict(int),
        'per_class_fn': defaultdict(int),
        'per_class_fp': defaultdict(int)
    }

    pred_dict = {p['image']: p for p in all_predictions}

    for gt_data in all_gt_data:
        img_name = gt_data['image']
        gt_classes = gt_data['classes']
        gt_boxes = gt_data.get('boxes', [])

        pred_data = pred_dict.get(img_name, {'classes': [], 'boxes': [], 'confs': []})
        pred_classes = pred_data['classes']
        pred_boxes = pred_data.get('boxes', [])

        stats['total_gt'] += len(gt_classes)
        stats['total_pred'] += len(pred_classes)

        if use_bbox_matching and len(gt_boxes) > 0 and len(pred_boxes) > 0:
            matched_pairs, unmatched_gt, unmatched_pred = match_predictions_to_gt(
               gt_classes, pred_classes, pred_boxes, gt_boxes, iou_threshold
            )

            for gt_cls, pred_cls in matched_pairs:
                y_true.append(gt_cls)
                y_pred.append(pred_cls)
                if gt_cls == pred_cls:
                    stats['matched'] += 1
                    stats['per_class_matches'][gt_cls] += 1

            for gt_cls in unmatched_gt:
                y_true.append(gt_cls)
                y_pred.append(-1)
                stats['false_negatives'] += 1
                stats['per_class_fn'][gt_cls] += 1

            for pred_cls in unmatched_pred:
                y_true.append(-1)
                y_pred.append(pred_cls)
                stats['false_positives'] += 1
                stats['per_class_fp'][pred_cls] += 1

        else:
            matched_pairs = match_by_class_only(gt_classes, pred_classes)

            for gt_cls, pred_cls in matched_pairs:
                y_true.append(gt_cls)
                y_pred.append(pred_cls)

                if gt_cls == pred_cls and gt_cls >= 0:
                    stats['matched'] += 1
                    stats['per_class_matches'][gt_cls] += 1
                elif gt_cls == -1:
                    stats['false_positives'] += 1
                    stats['per_class_fp'][pred_cls] += 1
                elif pred_cls == -1:
                    stats['false_negatives'] += 1
                    stats['per_class_fn'][gt_cls] += 1

    return y_true, y_pred, stats


def print_matching_statistics(stats, class_names):
    """Stampa statistiche matching dettagliate"""
    print("\n" + "="*80)
    print("📊 MATCHING STATISTICS")
    print("="*80)
    print(f"Total GT instances:      {stats['total_gt']}")
    print(f"Total Pred instances:    {stats['total_pred']}")
    print(f"Matched (TP):            {stats['matched']}")
    print(f"False Negatives (FN):    {stats['false_negatives']}")
    print(f"False Positives (FP):    {stats['false_positives']}")

    if stats['total_gt'] > 0:
        recall = stats['matched'] / stats['total_gt']
        print(f"Detection Recall:        {recall:.4f} ({recall*100:.2f}%)")

    if stats['total_pred'] > 0:
        precision = stats['matched'] / stats['total_pred']
        print(f"Detection Precision:     {precision:.4f} ({precision*100:.2f}%)")

    if stats['per_class_matches'] or stats['per_class_fn'] or stats['per_class_fp']:
        print("\n" + "-"*80)
        print("Per-Class Detection Statistics:")
        print("-"*80)
        print(f"{'Strumento':<30} {'Matched':>10} {'FN':>10} {'FP':>10}")
        print("-"*80)

        all_classes = set(stats['per_class_matches'].keys()) | \
                      set(stats['per_class_fn'].keys()) | \
                      set(stats['per_class_fp'].keys())

        for cls in sorted(all_classes):
            if cls >= 0 and cls < len(class_names):
                name = class_names[cls]
                matches = stats['per_class_matches'].get(cls, 0)
                fn = stats['per_class_fn'].get(cls, 0)
                fp = stats['per_class_fp'].get(cls, 0)
                print(f"{name:<30} {matches:>10} {fn:>10} {fp:>10}")

    print("="*80)


# ============================================================================
# FUNZIONI HELPER
# ============================================================================

def get_color_to_class_mapping():
    """Mappa colori RGB -> class_id"""
    return {
        (0, 0, 0): 0,
        (0, 0, 255): 1,
        (0, 255, 255): 2,
        (255, 255, 0): 4,
        (0, 255, 0): 5,
        (255, 0, 0): 6
    }
def apply_custom_colors_to_segmentation(pred_folder, image_folder, class_colors, class_names):
    """
    Applica colori personalizzati alle predizioni YOLO di segmentazione.
    Mostra: maschere colorate + bounding box + nome classe + confidence
    """
    import os
    import cv2
    import numpy as np
    from pathlib import Path

    print(f"\n  🎨 Applicazione colori personalizzati a segmentazione...")

    output_folder = os.path.join(pred_folder, "custom_colors")
    labels_folder = os.path.join(pred_folder, "labels")

    os.makedirs(output_folder, exist_ok=True)

    if not os.path.exists(labels_folder):
        print(f"  ⚠️ Cartella labels non trovata: {labels_folder}")
        return

    processed = 0
    image_files = [f for f in os.listdir(image_folder)
                   if f.lower().endswith(('.jpg', '.png', '.jpeg'))]

    for img_file in image_files:
        img_name = Path(img_file).stem
        img_path = os.path.join(image_folder, img_file)
        label_path = os.path.join(labels_folder, f"{img_name}.txt")

        img = cv2.imread(img_path)
        if img is None:
            continue

        h, w = img.shape[:2]

        if not os.path.exists(label_path):
            output_path = os.path.join(output_folder, f"{img_name}.jpg")
            cv2.imwrite(output_path, img)
            continue

        # Overlay per le maschere
        overlay = img.copy()

        with open(label_path, 'r') as f:
            predictions = f.readlines()

        for pred in predictions:
            parts = pred.strip().split()
            if len(parts) < 5:
                continue

            class_id = int(float(parts[0]))

            # Colore personalizzato
            color_key = class_id + 1
            color = class_colors.get(color_key, (255, 255, 255))

            # Nome classe e confidence (per etichetta)
            if class_id < len(class_names):
                class_name = class_names[class_id]
            else:
                class_name = f"Class {class_id}"

            # Se è segmentazione (polygon con molti punti)
            if len(parts) > 6:
                coords = list(map(float, parts[1:]))

                # Converti coordinate in pixel
                polygon_points = []
                for i in range(0, len(coords), 2):
                    if i+1 < len(coords):
                        x_pixel = int(coords[i] * w)
                        y_pixel = int(coords[i+1] * h)
                        polygon_points.append([x_pixel, y_pixel])

                if len(polygon_points) > 2:
                    polygon = np.array([polygon_points], dtype=np.int32)

                    # 1. DISEGNA MASCHERA COLORATA
                    mask = np.zeros((h, w), dtype=np.uint8)
                    cv2.fillPoly(mask, polygon, 255)
                    overlay[mask == 255] = color

                    # 2. CALCOLA BOUNDING BOX dal poligono
                    x_coords = [p[0] for p in polygon_points]
                    y_coords = [p[1] for p in polygon_points]
                    x1, y1 = min(x_coords), min(y_coords)
                    x2, y2 = max(x_coords), max(y_coords)

                    # 3. DISEGNA BOUNDING BOX
                    cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)

                    # 4. DISEGNA CONTORNO DELLA MASCHERA
                    cv2.polylines(img, [polygon], isClosed=True,
                                 color=color, thickness=2)

                    # 5. ETICHETTA CON NOME CLASSE (senza confidence per segmentation)
                    label = class_name

            else:
                # Detection (bounding box classico)
                x_center, y_center, box_w, box_h = map(float, parts[1:5])
                conf = float(parts[5]) if len(parts) > 5 else 1.0

                x1 = int((x_center - box_w/2) * w)
                y1 = int((y_center - box_h/2) * h)
                x2 = int((x_center + box_w/2) * w)
                y2 = int((y_center + box_h/2) * h)

                # Disegna bounding box
                cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)

                # Etichetta con confidence
                label = f"{class_name} {conf:.2f}"

            # DISEGNA ETICHETTA (stile YOLO)
            (text_w, text_h), baseline = cv2.getTextSize(
                label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 2)

            # Background rettangolare per il testo
            cv2.rectangle(img,
                         (x1, y1 - text_h - baseline - 8),
                         (x1 + text_w + 8, y1),
                         color, -1)

            # Testo in nero
            cv2.putText(img, label,
                       (x1 + 4, y1 - baseline - 4),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.5,
                       (0, 0, 0), 2)

        # Blend maschere con trasparenza
        alpha = 0.4
        result = cv2.addWeighted(overlay, alpha, img, 1 - alpha, 0)

        # Salva
        output_path = os.path.join(output_folder, f"{img_name}.jpg")
        cv2.imwrite(output_path, result)
        processed += 1

    print(f"  ✓ Applicati colori personalizzati a {processed} immagini")
    print(f"  📂 Salvate in: {output_folder}")
def extract_all_bboxes_from_mask(binary_mask, img_width, img_height, min_area=500):
    """Estrae TUTTE le bounding box da una maschera binaria"""

    kernel = np.ones((5, 5), np.uint8)

    binary_mask = cv2.morphologyEx(binary_mask.astype(np.uint8), cv2.MORPH_CLOSE, kernel)
    contours, _ = cv2.findContours(
        binary_mask.astype(np.uint8),
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    bboxes = []

    for contour in contours:

        area = cv2.contourArea(contour)
        if area < min_area:
            continue

        x, y, w, h = cv2.boundingRect(contour)

        x_center = (x + w / 2) / img_width
        y_center = (y + h / 2) / img_height
        width = w / img_width
        height = h / img_height

        bboxes.append([x_center, y_center, width, height])

    return bboxes


def convert_colored_mask_to_yolo_with_boxes(mask_path, output_txt_path, is_dataset_1_or_7_or_5=False):
    """Converte maschera RGB in YOLO DETECTION format (bbox only)"""
    if not os.path.exists(mask_path):
        return [], []

    mask = np.array(Image.open(mask_path).convert("RGB"))
    img_height, img_width = mask.shape[:2]

    color_to_class = get_color_to_class_mapping()

    yolo_labels = []
    classes_found = []
    boxes_found = []

    for color_rgb, class_id in color_to_class.items():
        if class_id == 0:
            continue

        match = np.all(mask == color_rgb, axis=-1)

        if is_dataset_1_or_7_or_5 and color_rgb == (255, 255, 0):
            if np.any(match):
                class_id = 2

        if not np.any(match):
            continue

        bboxes = extract_all_bboxes_from_mask(match, img_width, img_height, min_area=100)

        yolo_class_id = class_id - 1

        for bbox in bboxes:
            x_center, y_center, width, height = bbox

            yolo_labels.append(
                f"{yolo_class_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}"
            )

            classes_found.append(yolo_class_id)
            boxes_found.append(bbox)

    if yolo_labels:
        os.makedirs(os.path.dirname(output_txt_path), exist_ok=True)
        with open(output_txt_path, 'w') as f:
            f.write('\n'.join(yolo_labels))

    return classes_found, boxes_found


def create_yolo_labels_from_gt_with_boxes(image_folder, gt_folder, output_labels_folder, dataset_name):
    """Crea label YOLO e salva bbox GT per matching"""
    print(f"Convertendo GT in YOLO labels per {dataset_name}...")

    is_dataset_1_or_7 = "dataset_1" in dataset_name or "dataset_7" in dataset_name or "dataset_5" in dataset_name

    image_files = [f for f in os.listdir(image_folder)
                   if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

    converted = 0
    all_gt_data = []
    image_names = []

    for img_file in image_files:
        img_name = Path(img_file).stem
        mask_path = os.path.join(gt_folder, f"{img_name}.png")

        if not os.path.exists(mask_path):
            found = False
            if os.path.isdir(gt_folder):
                for subdir in os.listdir(gt_folder):
                    if subdir == "Other_labels":
                        continue
                    subdir_path = os.path.join(gt_folder, subdir)
                    if os.path.isdir(subdir_path):
                        mask_path_alt = os.path.join(subdir_path, f"{img_name}.png")
                        if os.path.exists(mask_path_alt):
                            mask_path = mask_path_alt
                            found = True
                            break
            if not found:
                continue

        output_txt = os.path.join(output_labels_folder, f"{img_name}.txt")
        classes, boxes = convert_colored_mask_to_yolo_with_boxes(
            mask_path, output_txt, is_dataset_1_or_7
        )

        if classes:
            converted += 1
            all_gt_data.append({
                'image': img_name,
                'classes': classes,
                'boxes': boxes
            })
            image_names.append(img_name)

    print(f"  ✓ Convertite {converted}/{len(image_files)} maschere")
    return all_gt_data, image_names


def parse_yolo_predictions_with_boxes(label_file):
    """Legge predizioni YOLO con bbox"""
    classes = []
    boxes = []
    confs = []

    if not os.path.exists(label_file):
        return classes, boxes, confs

    with open(label_file, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 5:
                continue

            cls = int(float(parts[0]))
            x, y, w, h = map(float, parts[1:5])
            conf = float(parts[5]) if len(parts) > 5 else 1.0

            classes.append(cls)
            boxes.append([x, y, w, h])
            confs.append(conf)

    return classes, boxes, confs


def merge_overlapping_boxes(classes, boxes, confs, iou_thresh=0.5):
    """Unisce predizioni frammentate della stessa classe"""
    merged_classes = []
    merged_boxes = []
    merged_confs = []

    for cls in np.unique(classes):
        idxs = [i for i, c in enumerate(classes) if c == cls]
        cls_boxes = np.array([boxes[i] for i in idxs])
        cls_confs = np.array([confs[i] for i in idxs])

        order = np.argsort(-cls_confs)
        cls_boxes = cls_boxes[order]
        cls_confs = cls_confs[order]

        kept = []
        for i, box in enumerate(cls_boxes):
            if any(iou(box, cls_boxes[j]) > iou_thresh for j in kept):
                continue
            kept.append(i)

        for i in kept:
            merged_classes.append(cls)
            merged_boxes.append(cls_boxes[i])
            merged_confs.append(cls_confs[i])

    return merged_classes, merged_boxes, merged_confs


def collect_predictions_with_boxes(pred_folder, image_names):
    """Raccoglie TUTTE le predizioni con bbox"""
    all_predictions = []

    for img_name in image_names:
        label_file = os.path.join(pred_folder, "labels", f"{img_name}.txt")

        classes, boxes, confs = parse_yolo_predictions_with_boxes(label_file)

        if classes:
            classes, boxes, confs = merge_overlapping_boxes(classes, boxes, confs, iou_thresh=0.5)
            classes = [int(c) for c in classes]
            confs = [float(c) for c in confs]
        else:
            classes, boxes, confs = [], [], []

        all_predictions.append({
            'image': img_name,
            'classes': classes,
            'boxes': boxes,
            'confs': confs
        })

    return all_predictions
CLASS_COLORS = {
    0: (0, 0, 0),          # background = nero
    1: (0, 0, 255),        # Large_Needle_Driver = rosso (BGR)
    2: (0, 255, 0),        # Prograsp_Forceps = verde (BGR)
    3: (255, 0, 0),        # Bipolar_Forceps = blu (BGR)
    4: (0, 255, 255),      # Grasping_Retractor = giallo (BGR)
    5: (255, 0, 255),      # Maryland_Bipolar_Forceps = magenta (BGR)
    6: (255, 255, 0),      # Monopolar_Curved_Scissors = ciano (BGR)
    7: (0, 165, 255),      # Vessel_Sealer = arancione (BGR)
}

# ============================================================================
# PROCESS DATASET - MODIFICATO PER AGGIUNGERE GT OVERLAY
# ============================================================================

def process_dataset(model, image_folder, gt_folder, dataset_name,
                   imgsz, conf, base_output_dir, temp_labels_dir, class_names):
    """Processa dataset con GT overlay sulle predizioni"""
    print(f"\n{'='*70}")
    print(f"DATASET: {dataset_name}")
    print(f"Images: {image_folder}")
    print(f"GT: {gt_folder}")
    print(f"{'='*70}")

    if not os.path.exists(image_folder):
        print(f"  ⚠️ Cartella immagini non trovata: {image_folder}")
        return None, None, None, None, None

    # Step 1: Crea YOLO labels con bbox
    gt_labels_folder = os.path.join(temp_labels_dir, dataset_name)
    all_gt_data, image_names = create_yolo_labels_from_gt_with_boxes(
        image_folder, gt_folder, gt_labels_folder, dataset_name
    )

    if not all_gt_data:
        print(f"  ⚠️ Nessuna GT trovata")
        return None, None, None, None, None

    # Step 2: Inferenza YOLO
    print(f"  Inferenza YOLO su {len(image_names)} immagini...")
    output_name = f"predict_{dataset_name}"

    start_total = time.time()

    results = model.predict(
        source=image_folder,
        imgsz=imgsz,
        save=True,
        save_txt=True,
        save_conf=True,
        project=base_output_dir,
        name=output_name,
        conf=conf,
        verbose=False,
        stream=True
    )

    inference_times = []
    processed_results = []

    for result in results:
        if hasattr(result, 'speed'):
            total_time = sum(result.speed.values())
            inference_times.append(total_time)
        else:
            inference_times.append(0.0)
        processed_results.append(result)

    total_time = time.time() - start_total

    if all(t == 0.0 for t in inference_times):
        avg_per_image = (total_time * 1000) / len(image_names)
        inference_times = [avg_per_image] * len(image_names)

    avg_time = np.mean(inference_times)
    min_time = np.min(inference_times)
    max_time = np.max(inference_times)
    std_time = np.std(inference_times)

    print(f"\n  ⏱️  TIMING INFERENZA:")
    print(f"     Tempo totale: {total_time:.2f}s")
    print(f"     Tempo medio per immagine: {avg_time:.2f}ms")
    print(f"     Min: {min_time:.2f}ms | Max: {max_time:.2f}ms | Std: {std_time:.2f}ms")
    print(f"     FPS: {1000/avg_time:.2f}")

    pred_folder = os.path.join(base_output_dir, output_name)
    apply_custom_colors_to_segmentation(pred_folder,image_folder,CLASS_COLORS,class_names)

    # Step 3: NUOVO - Overlay GT boxes sulle predizioni
    overlay_gt_boxes_on_predictions(pred_folder, image_folder, all_gt_data, class_names)

    # Step 4: Raccogli predizioni con bbox
    all_predictions = collect_predictions_with_boxes(pred_folder, image_names)

    # Step 5: Analizza distribuzione strumenti
    analyze_instrument_distribution(all_gt_data, all_predictions, class_names)

    # Step 6: Matching e calcolo metriche
    print(f"\n  🔄 Matching GT-Predictions con IoU threshold=0.5...")
    y_true, y_pred, match_stats = compute_metrics_with_matching(
        all_gt_data, all_predictions,
        use_bbox_matching=False,
        iou_threshold=0.4
    )

    print(f"  ✓ Matching completato: {match_stats['matched']} matched, "
          f"{match_stats['false_negatives']} FN, {match_stats['false_positives']} FP")

    # Step 7: Calcola confidence media
    all_confs = []
    for pred in all_predictions:
        all_confs.extend(pred['confs'])
    avg_conf = np.mean(all_confs) if all_confs else 0.0

    print(f"\n  ✓ Immagini: {len(image_names)} | Conf media: {avg_conf:.3f}")
    print_matching_statistics(match_stats, class_names)

    timing_data = {
        'image_names': image_names,
        'inference_times_ms': inference_times,
        'avg_time_ms': avg_time,
        'min_time_ms': min_time,
        'max_time_ms': max_time,
        'std_time_ms': std_time,
        'fps': 1000/avg_time,
        'total_time_s': total_time
    }

    return y_true, y_pred, all_confs, match_stats, timing_data


# ============================================================================
# TIMING REPORTS
# ============================================================================

def save_timing_report(all_timing_data, global_output_dir, dataset_results):
    """Salva report dettagliato dei tempi di inferenza"""
    timing_report_path = os.path.join(global_output_dir, "inference_timing_report.txt")

    all_times = []
    for data in all_timing_data.values():
        all_times.extend(data['inference_times_ms'])

    global_avg = np.mean(all_times)
    global_min = np.min(all_times)
    global_max = np.max(all_times)
    global_std = np.std(all_times)
    global_fps = 1000 / global_avg

    with open(timing_report_path, 'w') as f:
        f.write("="*80 + "\n")
        f.write("⏱️  INFERENCE TIMING REPORT\n")
        f.write("="*80 + "\n\n")

        f.write("STATISTICHE GLOBALI\n")
        f.write("-"*80 + "\n")
        f.write(f"Immagini totali:        {len(all_times)}\n")
        f.write(f"Tempo medio:            {global_avg:.2f} ms\n")
        f.write(f"Tempo minimo:           {global_min:.2f} ms\n")
        f.write(f"Tempo massimo:          {global_max:.2f} ms\n")
        f.write(f"Deviazione standard:    {global_std:.2f} ms\n")
        f.write(f"FPS medio:              {global_fps:.2f}\n")
        f.write(f"Tempo totale:           {sum([d['total_time_s'] for d in all_timing_data.values()]):.2f} s\n")
        f.write("\n")

        f.write("="*80 + "\n")
        f.write("STATISTICHE PER DATASET\n")
        f.write("="*80 + "\n\n")

        for dataset_name, timing_data in all_timing_data.items():
            f.write(f"--- {dataset_name} ---\n")
            f.write(f"Immagini:               {len(timing_data['image_names'])}\n")
            f.write(f"Tempo medio:            {timing_data['avg_time_ms']:.2f} ms\n")
            f.write(f"Tempo minimo:           {timing_data['min_time_ms']:.2f} ms\n")
            f.write(f"Tempo massimo:          {timing_data['max_time_ms']:.2f} ms\n")
            f.write(f"Deviazione standard:    {timing_data['std_time_ms']:.2f} ms\n")
            f.write(f"FPS:                    {timing_data['fps']:.2f}\n")
            f.write(f"Tempo totale:           {timing_data['total_time_s']:.2f} s\n")
            f.write("\n")

        f.write("="*80 + "\n")
        f.write("DETTAGLIO PER IMMAGINE (prime 10 per dataset)\n")
        f.write("="*80 + "\n\n")

        for dataset_name, timing_data in all_timing_data.items():
            f.write(f"--- {dataset_name} ---\n")
            for i, (img_name, time_ms) in enumerate(zip(
                timing_data['image_names'][:10],
                timing_data['inference_times_ms'][:10]
            )):
                f.write(f"  {img_name}: {time_ms:.2f} ms\n")
            if len(timing_data['image_names']) > 10:
                f.write(f"  ... (altre {len(timing_data['image_names'])-10} immagini)\n")
            f.write("\n")

    print(f"💾 Timing report salvato: {timing_report_path}")
    save_timing_csv(all_timing_data, global_output_dir)
    plot_timing_distribution(all_timing_data, global_output_dir)


def save_timing_csv(all_timing_data, global_output_dir):
    """Salva timing in CSV per analisi"""
    csv_path = os.path.join(global_output_dir, "inference_timing.csv")

    with open(csv_path, 'w') as f:
        f.write("dataset,image_name,inference_time_ms\n")

        for dataset_name, timing_data in all_timing_data.items():
            for img_name, time_ms in zip(
                timing_data['image_names'],
                timing_data['inference_times_ms']
            ):
                f.write(f"{dataset_name},{img_name},{time_ms:.4f}\n")

    print(f"💾 Timing CSV salvato: {csv_path}")


def plot_timing_distribution(all_timing_data, global_output_dir):
    """Genera grafici distribuzione tempi"""
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    dataset_names = list(all_timing_data.keys())
    dataset_times = [all_timing_data[ds]['inference_times_ms'] for ds in dataset_names]

    axes[0].boxplot(dataset_times, labels=dataset_names)
    axes[0].set_ylabel('Tempo Inferenza (ms)', fontsize=12)
    axes[0].set_xlabel('Dataset', fontsize=12)
    axes[0].set_title('Distribuzione Tempi per Dataset', fontsize=14, fontweight='bold')
    axes[0].tick_params(axis='x', rotation=45)
    axes[0].grid(True, alpha=0.3)

    all_times = []
    for data in all_timing_data.values():
        all_times.extend(data['inference_times_ms'])

    axes[1].hist(all_times, bins=30, edgecolor='black', alpha=0.7)
    axes[1].axvline(np.mean(all_times), color='red', linestyle='--',
                    label=f'Media: {np.mean(all_times):.2f}ms')
    axes[1].set_xlabel('Tempo Inferenza (ms)', fontsize=12)
    axes[1].set_ylabel('Frequenza', fontsize=12)
    axes[1].set_title('Distribuzione Globale Tempi', fontsize=14, fontweight='bold')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plot_path = os.path.join(global_output_dir, "timing_distribution.png")
    plt.savefig(plot_path, dpi=300)
    plt.close()

    print(f"📊 Grafici timing salvati: {plot_path}")


def find_all_datasets(root_dir, gt_type='TypeSegmentation'):
    """Trova tutti i dataset con cartelle TypeSegmentation"""
    dataset_paths = []

    for root, dirs, files in os.walk(root_dir):
        if gt_type in dirs:
            type_seg = os.path.join(root, gt_type)
            gt_folder = type_seg
            dataset_name = Path(root).name
            dataset_paths.append((gt_folder, dataset_name))
            print(f"✓ Dataset: {dataset_name} | GT: {Path(gt_folder).name}")

    return dataset_paths


def plot_confusion_matrix(y_true, y_pred, class_names, save_path):
    """Genera confusion matrix"""
    valid_pairs = [(gt, pred) for gt, pred in zip(y_true, y_pred)
                   if gt >= 0 and pred >= 0]

    if not valid_pairs:
        print("WARNING: Nessuna predizione valida per confusion matrix")
        return

    filtered_y_true, filtered_y_pred = zip(*valid_pairs)

    unique_labels = sorted(set(filtered_y_true) | set(filtered_y_pred))
    label_names = [class_names[i] if i < len(class_names) else f"Class_{i}"
                   for i in unique_labels]

    cm = confusion_matrix(filtered_y_true, filtered_y_pred, labels=unique_labels)

    plt.figure(figsize=(12, 10))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=label_names, yticklabels=label_names,
                cbar_kws={'label': 'Count'})
    plt.title('Confusion Matrix', fontsize=14, fontweight='bold')
    plt.ylabel('True Label', fontsize=12)
    plt.xlabel('Predicted Label', fontsize=12)
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    plt.close()
    print(f"Confusion matrix: {save_path}")


# ============================================================================
# MAIN EVALUATION PIPELINE
# ============================================================================

def generate_evaluation_report(model_path, gt_root_dirs, image_folders, class_names,
                               gt_type='TypeSegmentation',
                               imgsz=1024, conf=0.45,
                               base_output_dir="runs"):
    """Pipeline completa con statistiche dettagliate per strumento"""

    if isinstance(gt_root_dirs, str):
        gt_root_dirs = [gt_root_dirs]

    print("="*70)
    print("YOLO EVALUATION PIPELINE - CON GT BOUNDING BOXES OVERLAY")
    print("="*70)
    print(f"Modello: {model_path}")
    print(f"GT Type: {gt_type}")
    print(f"Imgsz: {imgsz} | Conf: {conf}")
    print("="*70 + "\n")

    print("📦 Caricamento modello YOLO...")
    model = YOLO(model_path)

    all_gt_datasets = []
    for root_dir in gt_root_dirs:
        datasets = find_all_datasets(root_dir, gt_type)
        all_gt_datasets.extend(datasets)

    if not all_gt_datasets:
        print("❌ ERRORE: Nessun dataset GT trovato!")
        return

    print(f"\n📁 Dataset GT trovati: {len(all_gt_datasets)}")

    if isinstance(image_folders, dict):
        dataset_to_images = image_folders
    elif isinstance(image_folders, list):
        dataset_to_images = {}
        for gt_folder, dataset_name in all_gt_datasets:
            matched = False
            for img_folder in image_folders:
                if dataset_name in img_folder or Path(img_folder).name == dataset_name:
                    dataset_to_images[dataset_name] = img_folder
                    matched = True
                    break
            if not matched:
                dataset_to_images[dataset_name] = image_folders[0]
    else:
        dataset_to_images = {ds_name: image_folders for _, ds_name in all_gt_datasets}

    print("\nMapping Dataset -> Immagini:")
    for ds_name, img_path in dataset_to_images.items():
        print(f"  {ds_name} -> {img_path}")
    print()

    temp_labels_dir = os.path.join(base_output_dir, "temp_yolo_labels")
    os.makedirs(temp_labels_dir, exist_ok=True)

    all_y_true = []
    all_y_pred = []
    all_confidences = []
    dataset_results = {}
    all_timing_data = {}

    for gt_folder, dataset_name in all_gt_datasets:
        image_folder = dataset_to_images.get(dataset_name)

        if image_folder is None:
            print(f"⚠️ Nessuna cartella immagini per {dataset_name}, skip")
            continue

        result = process_dataset(
            model, image_folder, gt_folder, dataset_name,
            imgsz, conf, base_output_dir, temp_labels_dir, class_names
        )

        if result[0] is not None:
            y_true, y_pred, confidences, match_stats, timing_data = result

            all_y_true.extend(y_true)
            all_y_pred.extend(y_pred)
            all_confidences.extend(confidences)

            dataset_results[dataset_name] = {
                'y_true': y_true,
                'y_pred': y_pred,
                'confidences': confidences,
                'match_stats': match_stats
            }
            all_timing_data[dataset_name] = timing_data

    if not all_y_true:
        print("❌ Nessun risultato ottenuto!")
        return

    print("\n" + "="*70)
    print("📊 CLASSIFICATION REPORT - GLOBALE")
    print("="*70)
    print(f"Campioni totali (istanze): {len(all_y_true)}")
    print(f"Confidence media: {np.mean(all_confidences):.4f}")
    print("="*70 + "\n")

    valid_pairs = [(gt, pred) for gt, pred in zip(all_y_true, all_y_pred)
                   if gt >= 0 and pred >= 0]

    if not valid_pairs:
        print("❌ Nessun campione valido!")
        return

    filtered_y_true, filtered_y_pred = zip(*valid_pairs)
    unique_classes = sorted(set(filtered_y_true))

    report = classification_report(
        filtered_y_true,
        filtered_y_pred,
        labels=unique_classes,
        target_names=[class_names[i] for i in unique_classes],
        digits=4,
        zero_division=0
    )
    print(report)

    print("\n" + "="*70)
    print("📊 STATISTICHE GLOBALI PER STRUMENTO")
    print("="*70)

    global_per_class_stats = compute_per_class_statistics(all_y_true, all_y_pred, class_names)
    overall_accuracy = accuracy_score(filtered_y_true, filtered_y_pred)

    print_per_class_report(global_per_class_stats, overall_accuracy)

    global_output_dir = os.path.join(base_output_dir, "evaluation_report")
    os.makedirs(global_output_dir, exist_ok=True)

    report_path = os.path.join(global_output_dir, "classification_report.txt")
    with open(report_path, 'w') as f:
        f.write("YOLO CLASSIFICATION REPORT - CON GT BOUNDING BOXES OVERLAY\n")
        f.write("="*70 + "\n")
        f.write(f"Modello: {model_path}\n")
        f.write(f"GT Type: {gt_type}\n")
        f.write(f"Imgsz: {imgsz} | Conf: {conf}\n")
        f.write(f"Dataset: {len(dataset_results)}\n")
        f.write(f"Istanze totali: {len(all_y_true)}\n")
        f.write(f"Istanze valide: {len(filtered_y_true)}\n")
        f.write(f"Overall Accuracy: {overall_accuracy:.4f} ({overall_accuracy*100:.2f}%)\n")
        f.write(f"Confidence media: {np.mean(all_confidences):.4f}\n")
        f.write("="*70 + "\n\n")

        f.write("SKLEARN CLASSIFICATION REPORT\n")
        f.write("-"*70 + "\n")
        f.write(report)

        f.write("\n\nMETRICHE DETTAGLIATE PER STRUMENTO (GLOBALI)\n")
        f.write("="*70 + "\n")

        for class_id in sorted(global_per_class_stats.keys()):
            stats = global_per_class_stats[class_id]
            f.write(f"\n{stats['name']} (ID: {class_id})\n")
            f.write("-" * 70 + "\n")
            f.write(f"  Precision:     {stats['precision']:.4f}\n")
            f.write(f"  Recall:        {stats['recall']:.4f}\n")
            f.write(f"  Sensitivity:   {stats['sensitivity']:.4f}\n")
            f.write(f"  Specificity:   {stats['specificity']:.4f}\n")
            f.write(f"  F1-Score:      {stats['f1']:.4f}\n")
            f.write(f"  Accuracy:      {stats['accuracy']:.4f}\n")
            f.write(f"  GT Instances:  {stats['total_gt']}\n")
            f.write(f"  Pred Instances:{stats['total_pred']}\n")
            f.write(f"  TP: {stats['tp']:4d}  |  FP: {stats['fp']:4d}  |  ")
            f.write(f"FN: {stats['fn']:4d}  |  TN: {stats['tn']:4d}\n")

        avg_precision = np.mean([s['precision'] for s in global_per_class_stats.values()])
        avg_recall = np.mean([s['recall'] for s in global_per_class_stats.values()])
        avg_f1 = np.mean([s['f1'] for s in global_per_class_stats.values()])
        avg_sensitivity = np.mean([s['sensitivity'] for s in global_per_class_stats.values()])
        avg_specificity = np.mean([s['specificity'] for s in global_per_class_stats.values()])

        f.write("\n" + "="*70 + "\n")
        f.write("MACRO AVERAGES (GLOBALI)\n")
        f.write("-"*70 + "\n")
        f.write(f"Precision:   {avg_precision:.4f}\n")
        f.write(f"Recall:      {avg_recall:.4f}\n")
        f.write(f"Sensitivity: {avg_sensitivity:.4f}\n")
        f.write(f"Specificity: {avg_specificity:.4f}\n")
        f.write(f"F1-Score:    {avg_f1:.4f}\n")

        f.write("\n\n" + "="*70 + "\n")
        f.write("STATISTICHE PER DATASET\n")
        f.write("="*70 + "\n")

        for ds_name, res in dataset_results.items():
            f.write(f"\n--- {ds_name} ---\n")
            f.write(f"Istanze GT:   {len([y for y in res['y_true'] if y >= 0])}\n")
            f.write(f"Istanze Pred: {len([y for y in res['y_pred'] if y >= 0])}\n")

            match_stats = res['match_stats']
            f.write(f"Matched (TP): {match_stats['matched']}\n")
            f.write(f"FN:           {match_stats['false_negatives']}\n")
            f.write(f"FP:           {match_stats['false_positives']}\n")

            gt_counter = Counter([y for y in res['y_true'] if y >= 0])
            pred_counter = Counter([y for y in res['y_pred'] if y >= 0])

            f.write("\nDistribuzione Strumenti:\n")
            all_classes = sorted(set(gt_counter.keys()) | set(pred_counter.keys()))
            for cls in all_classes:
                if cls < len(class_names):
                    f.write(f"  {class_names[cls]}: GT={gt_counter.get(cls, 0)} Pred={pred_counter.get(cls, 0)}\n")
            f.write("\n")

    print(f"\n💾 Report salvato: {report_path}")

    cm_path = os.path.join(global_output_dir, "confusion_matrix.png")
    plot_confusion_matrix(filtered_y_true, filtered_y_pred, class_names, cm_path)

    print("\n" + "="*70)
    print("⏱️  GENERAZIONE TIMING REPORT")
    print("="*70)
    save_timing_report(all_timing_data, global_output_dir, dataset_results)

    with open(report_path, 'a') as f:
        f.write("\n\n" + "="*70 + "\n")
        f.write("⏱️  INFERENCE TIMING SUMMARY\n")
        f.write("="*70 + "\n")

        all_times = []
        for data in all_timing_data.values():
            all_times.extend(data['inference_times_ms'])

        f.write(f"Immagini totali:        {len(all_times)}\n")
        f.write(f"Tempo medio:            {np.mean(all_times):.2f} ms\n")
        f.write(f"FPS medio:              {1000/np.mean(all_times):.2f}\n")
        f.write(f"Tempo totale:           {sum([d['total_time_s'] for d in all_timing_data.values()]):.2f} s\n")
        f.write("\nPer dettagli completi vedi: inference_timing_report.txt\n")
        f.write("\n💡 Le immagini con GT boxes sovrapposte sono salvate in: runs/predict_*/with_gt_boxes/\n")

    print("\n✅ COMPLETATO!")
    print(f"📂 Risultati in: {global_output_dir}")
    print(f"🎨 Immagini con GT overlay in: runs/predict_*/with_gt_boxes/")

    return all_y_true, all_y_pred, all_confidences


# ============================================================================
# MAIN
# ============================================================================
if __name__ == "__main__":

    MODEL_PATH = "/content/drive/MyDrive/yolo/segment/instrument_seg/weights/best.pt"

    GT_ROOT_DIRS = [
        "/content/drive/MyDrive/yolo/instrument_2017_test/instrument_2017_test/instrument_dataset_1",
        #"/content/drive/MyDrive/yolo/instrument_2017_test/instrument_2017_test/instrument_dataset_2",
        #"/content/drive/MyDrive/yolo/instrument_2017_test/instrument_2017_test/instrument_dataset_3",
        #"/content/drive/MyDrive/yolo/instrument_2017_test/instrument_2017_test/instrument_dataset_4",
        #"/content/drive/MyDrive/yolo/instrument_2017_test/instrument_2017_test/instrument_dataset_5",
        #"/content/drive/MyDrive/yolo/instrument_2017_test/instrument_2017_test/instrument_dataset_6",
        #"/content/drive/MyDrive/yolo/instrument_2017_test/instrument_2017_test/instrument_dataset_7",
        "/content/drive/MyDrive/yolo/instrument_2017_test/instrument_2017_test/instrument_dataset_8",
    ]

    IMAGE_FOLDERS = {
        'instrument_dataset_1': '/content/drive/MyDrive/yolo/instrument_1_4_testing/instrument_dataset_1/left_frames',
        #'instrument_dataset_2': '/content/drive/MyDrive/yolo/instrument_1_4_testing/instrument_dataset_2/left_frames2',
        #'instrument_dataset_3': '/content/drive/MyDrive/yolo/instrument_1_4_testing/instrument_dataset_3/left_frames',
        #'instrument_dataset_4': '/content/drive/MyDrive/yolo/instrument_1_4_testing/instrument_dataset_4/left_frames',
        #'instrument_dataset_5': '/content/drive/MyDrive/yolo/instrument_5_8_testing/instrument_dataset_5/left_frames',
        #'instrument_dataset_6': '/content/drive/MyDrive/yolo/instrument_5_8_testing/instrument_dataset_6/left_frames',
        #'instrument_dataset_7': '/content/drive/MyDrive/yolo/instrument_5_8_testing/instrument_dataset_7/left_frames',
        'instrument_dataset_8': '/content/drive/MyDrive/yolo/instrument_5_8_testing/instrument_dataset_8/left_frames',
    }

    CLASS_NAMES = [
        'Large_Needle_Driver',
        'Forceps',
        'Grasping_Retractor',
        'Maryland_Bipolar_Forceps',
        'Monopolar_Curved_Scissors',
        'Vessel_Sealer'
    ]

    generate_evaluation_report(
        model_path=MODEL_PATH,
        gt_root_dirs=GT_ROOT_DIRS,
        image_folders=IMAGE_FOLDERS,
        class_names=CLASS_NAMES,
        gt_type='TypeSegmentationRescaled',
        imgsz=1024,
        conf=0.45
    )

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
YOLO EVALUATION PIPELINE - CON GT BOUNDING BOXES OVERLAY
Modello: /content/drive/MyDrive/yolo/segment/instrument_seg/weights/best.pt
GT Type: TypeSegmentationRescaled
Imgsz: 1024 | Conf: 0.45

📦 Caricamento modello YOLO...


KeyboardInterrupt: 

In [ ]:
pip install ultralytics

In [4]:
import os
import random
import numpy as np
from PIL import Image
from torch.utils.data import Dataset
import albumentations as A
import json
from albumentations.pytorch import ToTensorV2
import torch
from torchvision.transforms import ToTensor
from pycocotools.coco import COCO
from pycocotools import mask as maskUtils
class DatasetTest(Dataset):
    def __init__(self, image_dirs, coco, transform=None, top_n=None, min_area=None):
        """
        Dataset per immagini con annotazioni COCO.

        Args:
            image_dirs: Lista di directory contenenti le immagini
            coco_file: Path al file annotations.json in formato COCO
            transform: Trasformazioni Albumentations (opzionale)
            top_n: Numero massimo di bbox da restituire (le più grandi)
            min_area: Area minima per filtrare le bbox
        """
        self.image_dirs = image_dirs if isinstance(image_dirs, list) else [image_dirs]
        self.transform = transform
        self.top_n = top_n
        self.min_area = min_area

        # Carica COCO
        with open(coco, "r") as f:
            self.coco_data = json.load(f)
        self.coco = COCO(coco)

        # Crea mappatura filename -> image_id per ricerca veloce
        self.filename_to_id = {
            img["file_name"]: img["id"]
            for img in self.coco_data["images"]
        }

        # Trova tutti i percorsi delle immagini
        self.image_paths = []
        for img_dir in self.image_dirs:
            if not os.path.exists(img_dir):
                print(f"⚠️  Directory non trovata: {img_dir}")
                continue

            for filename in os.listdir(img_dir):
                if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
                    full_path = os.path.join(img_dir, filename)
                    # Verifica che l'immagine sia nel file COCO
                    if filename in self.filename_to_id:
                        self.image_paths.append(full_path)
                    else:
                        print(f"⚠️  Immagine non nel COCO: {filename}")

        print(f"✓ Trovate {len(self.image_paths)} immagini con annotazioni")

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        # Carica immagine
        img_path = self.image_paths[idx]
        image = np.array(Image.open(img_path).convert("RGB"))

        # Estrai filename
        filename = os.path.basename(img_path)

        # Trova image_id
        img_id = self.filename_to_id.get(filename)

        if img_id is None:
            print(f"⚠️  Nessuna annotazione per: {filename}")
            # Ritorna immagine senza bbox
            if self.transform:
                augmented = self.transform(image=image)
                image = augmented["image"]
            else:
                image = self._default_transform(image)
            return image, []

        # Carica annotazioni
        ann_ids = self.coco.getAnnIds(imgIds=[img_id])
        anns = self.coco.loadAnns(ann_ids)

        # Estrai bboxes: [x_min, y_min, width, height] (formato COCO)
        bboxes_with_area = []
        for ann in anns:
            x, y, w, h = ann["bbox"]
            area = ann.get("area", w * h)
            bboxes_with_area.append({
                "bbox": [x, y, w, h],
                "bbox_xyxy": [x, y, x + w, y + h],  # Formato [x1, y1, x2, y2]
                "area": area,
                "category_id": ann["category_id"]
            })

        # Filtra per area minima
        if self.min_area is not None:
            bboxes_with_area = [b for b in bboxes_with_area if b["area"] >= self.min_area]

        # Ordina per area (più grande prima) e prendi top N
        bboxes_with_area.sort(key=lambda x: x["area"], reverse=True)
        if self.top_n is not None:
            bboxes_with_area = bboxes_with_area[:self.top_n]

        # Estrai bbox in formato COCO [x, y, w, h]
        bboxes = [b["bbox"] for b in bboxes_with_area]

        # Applica trasformazioni
        if self.transform:
            # Albumentations richiede bbox in formato [x_min, y_min, x_max, y_max] normalizzato
            h, w = image.shape[:2]
            bbox_params = A.BboxParams(
                format='coco',  # COCO format: [x, y, width, height]
                label_fields=['category_ids'],
                min_area=0,
                min_visibility=0
            )

            transform_with_bbox = A.Compose(
                self.transform.transforms if hasattr(self.transform, 'transforms') else [self.transform],
                bbox_params=bbox_params
            )

            category_ids = [b["category_id"] for b in bboxes_with_area]


            augmented = transform_with_bbox(
                image=image,
                bboxes=bboxes,
                category_ids=category_ids
            )
            image = augmented["image"]
            bboxes = augmented["bboxes"]
        else:
            image = self._default_transform(image)

        return image, bboxes


In [ ]:
pip install ultralytics

In [11]:
import torch
import numpy as np
from ultralytics import YOLO
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import DataLoader
from matplotlib import pyplot as plt
import pandas as pd
import time
import cv2
import os


class YOLOInferenceWrapper:
    """Wrapper per inferenza YOLO"""

    def __init__(self, model_path, device='cuda'):
        self.model = YOLO(model_path)
        self.device = device
        self.model.to(device)

    def predict_from_tensor(self, image_tensor, conf=0.01, iou=0.5, imgsz=1024):
        if image_tensor.dim() == 3:
            image_tensor = image_tensor.unsqueeze(0)

        img_denorm = image_tensor * 0.5 + 0.5
        img_denorm = img_denorm.clamp(0, 1)

        img_np = img_denorm.squeeze(0).permute(1, 2, 0).cpu().numpy()
        img_np = (img_np * 255).astype(np.uint8)

        h, w = img_np.shape[:2]

        try:
            results = self.model.predict(
                source=img_np, conf=conf, iou=iou, imgsz=imgsz,
                save=False, verbose=False, device=self.device
            )
        except Exception as e:
            return np.zeros((h, w), dtype=np.uint8), None, None, None, None

        if results and len(results) > 0:
            result = results[0]
            if result.boxes is not None and len(result.boxes) > 0:
                boxes = result.boxes.xyxy.cpu().numpy()
                scores = result.boxes.conf.cpu().numpy()
                classes = result.boxes.cls.cpu().numpy().astype(int)

                if result.masks is not None:
                    masks = result.masks.data.cpu().numpy()
                    combined_mask = np.zeros((h, w), dtype=np.uint8)
                    for mask in masks:
                        mask_resized = cv2.resize(
                            mask.astype(np.float32), (w, h),
                            interpolation=cv2.INTER_LINEAR
                        )
                        combined_mask = np.maximum(
                            combined_mask,
                            (mask_resized > 0.5).astype(np.uint8)
                        )
                    return combined_mask, masks, boxes, scores, classes
                else:
                    return np.zeros((h, w), dtype=np.uint8), None, boxes, scores, classes

        return np.zeros((h, w), dtype=np.uint8), None, None, None, None


def calculate_iou_single(boxA, boxB):
    """Calcola IoU tra due singole bounding box"""
    xA1, yA1, xA2, yA2 = boxA[:4]
    xB1, yB1, xB2, yB2 = boxB[:4]

    x_left = max(xA1, xB1)
    y_top = max(yA1, yB1)
    x_right = min(xA2, xB2)
    y_bottom = min(yA2, yB2)

    if x_right <= x_left or y_bottom <= y_top:
        return 0.0

    inter_area = (x_right - x_left) * (y_bottom - y_top)
    areaA = (xA2 - xA1) * (yA2 - yA1)
    areaB = (xB2 - xB1) * (yB2 - yB1)
    union_area = areaA + areaB - inter_area

    return inter_area / union_area if union_area > 0 else 0.0


def calculate_dice_from_iou(iou):
    """Converte IoU in Dice: Dice = 2*IoU / (1+IoU)"""
    if iou == 0:
        return 0.0
    return (2.0 * iou) / (1.0 + iou)


def calculate_pixel_level_metrics(preds, gts, image_shape=(1024, 1024)):
    """
    Calcola metriche PIXEL-LEVEL da bounding boxes.

    Args:
        preds: lista di [x1, y1, x2, y2, score]
        gts: lista di [x1, y1, x2, y2]
        image_shape: (H, W) dimensioni immagine

    Returns:
        dict con TP, FP, FN, TN pixel, Sensitivity, Specificity, Dice pixel-level
    """
    h, w = image_shape
    total_pixels = h * w

    # Crea maschere binarie dalle bounding box
    pred_mask = np.zeros((h, w), dtype=np.uint8)
    gt_mask = np.zeros((h, w), dtype=np.uint8)

    # Riempi maschera predizioni
    for pred in preds:
        x1, y1, x2, y2 = map(int, pred[:4])
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(w, x2), min(h, y2)
        pred_mask[y1:y2, x1:x2] = 1

    # Riempi maschera GT
    for gt in gts:
        x1, y1, x2, y2 = map(int, gt[:4])
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(w, x2), min(h, y2)
        gt_mask[y1:y2, x1:x2] = 1

    # Calcola confusion matrix pixel-level
    tp_pixels = np.sum((pred_mask == 1) & (gt_mask == 1))
    fp_pixels = np.sum((pred_mask == 1) & (gt_mask == 0))
    fn_pixels = np.sum((pred_mask == 0) & (gt_mask == 1))
    tn_pixels = np.sum((pred_mask == 0) & (gt_mask == 0))

    # Verifica
    assert tp_pixels + fp_pixels + fn_pixels + tn_pixels == total_pixels

    # Sensitivity (Recall pixel-level) = TP / (TP + FN)
    sensitivity = tp_pixels / (tp_pixels + fn_pixels) if (tp_pixels + fn_pixels) > 0 else 0.0

    # Specificity = TN / (TN + FP)
    specificity = tn_pixels / (tn_pixels + fp_pixels) if (tn_pixels + fp_pixels) > 0 else 0.0

    # Precision pixel-level = TP / (TP + FP)
    precision_pixel = tp_pixels / (tp_pixels + fp_pixels) if (tp_pixels + fp_pixels) > 0 else 0.0

    # Dice pixel-level = 2*TP / (2*TP + FP + FN)
    dice_pixel = (2 * tp_pixels) / (2 * tp_pixels + fp_pixels + fn_pixels) \
        if (2 * tp_pixels + fp_pixels + fn_pixels) > 0 else 0.0

    # Accuracy pixel-level = (TP + TN) / Total
    accuracy_pixel = (tp_pixels + tn_pixels) / total_pixels

    # IoU pixel-level = TP / (TP + FP + FN)
    iou_pixel = tp_pixels / (tp_pixels + fp_pixels + fn_pixels) \
        if (tp_pixels + fp_pixels + fn_pixels) > 0 else 0.0

    return {
        'tp_pixels': int(tp_pixels),
        'fp_pixels': int(fp_pixels),
        'fn_pixels': int(fn_pixels),
        'tn_pixels': int(tn_pixels),
        'sensitivity_pixel': float(sensitivity),
        'specificity_pixel': float(specificity),
        'precision_pixel': float(precision_pixel),
        'dice_pixel': float(dice_pixel),
        'iou_pixel': float(iou_pixel),
        'accuracy_pixel': float(accuracy_pixel)
    }


def calculate_metrics_per_image(preds, gts, iou_threshold=0.5, image_shape=(1024, 1024)):
    """
    Calcola metriche BOX-LEVEL + PIXEL-LEVEL per una singola immagine.

    Returns:
        dict con metriche box-level E pixel-level
    """
    # ========== BOX-LEVEL METRICS ==========
    if len(preds) == 0 and len(gts) == 0:
        pixel_metrics = calculate_pixel_level_metrics(preds, gts, image_shape)
        return {
            'tp': 0, 'fp': 0, 'fn': 0,
            'sensitivity_box': 1.0,
            'specificity_box': 1.0,
            'mean_dice_box': 1.0,
            'matched_pairs': [],
            **pixel_metrics
        }

    if len(preds) == 0:
        pixel_metrics = calculate_pixel_level_metrics(preds, gts, image_shape)
        return {
            'tp': 0, 'fp': 0, 'fn': len(gts),
            'sensitivity_box': 0.0,
            'specificity_box': 1.0,
            'mean_dice_box': 0.0,
            'matched_pairs': [],
            **pixel_metrics
        }

    if len(gts) == 0:
        pixel_metrics = calculate_pixel_level_metrics(preds, gts, image_shape)
        return {
            'tp': 0, 'fp': len(preds), 'fn': 0,
            'sensitivity_box': 0.0,
            'specificity_box': 0.0,
            'mean_dice_box': 0.0,
            'matched_pairs': [],
            **pixel_metrics
        }

    # Calcola matrice IoU
    iou_matrix = np.zeros((len(preds), len(gts)))
    for i, pred in enumerate(preds):
        for j, gt in enumerate(gts):
            iou_matrix[i, j] = calculate_iou_single(pred, gt)

    # Greedy matching
    matches = []
    for i in range(len(preds)):
        for j in range(len(gts)):
            if iou_matrix[i, j] >= iou_threshold:
                matches.append((i, j, iou_matrix[i, j]))

    matches.sort(key=lambda x: x[2], reverse=True)

    matched_preds = set()
    matched_gts = set()
    matched_pairs = []
    dice_scores = []

    for pred_idx, gt_idx, iou_val in matches:
        if pred_idx not in matched_preds and gt_idx not in matched_gts:
            matched_preds.add(pred_idx)
            matched_gts.add(gt_idx)
            matched_pairs.append((pred_idx, gt_idx, iou_val))
            dice_scores.append(calculate_dice_from_iou(iou_val))

    tp = len(matched_pairs)
    fp = len(preds) - tp
    fn = len(gts) - tp

    # Box-level metrics
    sensitivity_box = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    specificity_box = 1.0 - (fp / len(preds)) if len(preds) > 0 else 1.0
    mean_dice_box = np.mean(dice_scores) if dice_scores else 0.0

    # ========== PIXEL-LEVEL METRICS ==========
    pixel_metrics = calculate_pixel_level_metrics(preds, gts, image_shape)

    return {
        # Box-level
        'tp': tp,
        'fp': fp,
        'fn': fn,
        'sensitivity_box': sensitivity_box,
        'specificity_box': specificity_box,
        'mean_dice_box': mean_dice_box,
        'matched_pairs': matched_pairs,
        # Pixel-level
        **pixel_metrics
    }


def compute_pr_curve_and_ap(all_preds, all_gts, iou_threshold=0.5):
    """Calcola curva Precision-Recall e AP"""
    if len(all_preds) == 0:
        return np.array([0]), np.array([0]), 0.0

    all_preds_sorted = sorted(all_preds, key=lambda x: x[4], reverse=True)

    gt_by_image = {}
    for gt in all_gts:
        img_id = gt[4]
        if img_id not in gt_by_image:
            gt_by_image[img_id] = []
        gt_by_image[img_id].append(gt[:4])

    matched_gt = {img_id: set() for img_id in gt_by_image.keys()}

    tp_list = []
    fp_list = []

    for pred in all_preds_sorted:
        px1, py1, px2, py2, score, img_id = pred

        if img_id not in gt_by_image:
            tp_list.append(0)
            fp_list.append(1)
            continue

        best_iou = 0.0
        best_gt_idx = -1

        for gt_idx, gt in enumerate(gt_by_image[img_id]):
            iou = calculate_iou_single([px1, py1, px2, py2], gt)
            if iou > best_iou:
                best_iou = iou
                best_gt_idx = gt_idx

        if best_iou >= iou_threshold and best_gt_idx not in matched_gt[img_id]:
            tp_list.append(1)
            fp_list.append(0)
            matched_gt[img_id].add(best_gt_idx)
        else:
            tp_list.append(0)
            fp_list.append(1)

    tp_cumsum = np.cumsum(tp_list)
    fp_cumsum = np.cumsum(fp_list)
    total_gt = len(all_gts)

    recalls = tp_cumsum / total_gt
    precisions = tp_cumsum / (tp_cumsum + fp_cumsum + 1e-10)

    ap = 0.0
    for t in np.linspace(0, 1, 11):
        if np.sum(recalls >= t) == 0:
            p = 0
        else:
            p = np.max(precisions[recalls >= t])
        ap += p / 11

    return recalls, precisions, ap


def refining(mask):
    """Post-processing morfologico"""
    mask = (mask * 255).astype(np.uint8)
    while mask.ndim > 2:
        mask = mask[0]
    kernel = np.ones((3, 3), np.uint8)
    mask_clean = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask_clean = cv2.morphologyEx(mask_clean, cv2.MORPH_CLOSE, kernel)
    mask_blurred = cv2.GaussianBlur(mask_clean, (5, 5), 0)
    return mask_blurred / 255

def collate_fn(batch):
    return [item[0] for item in batch], [item[1] for item in batch]


def run_yolo_inference_with_dice():
    """Script completo con Dice, Sensitivity, Specificity"""
    print("="*70)
    print("🚀 YOLO INFERENCE - DICE + SENSITIVITY/SPECIFICITY")
    print("="*70)

    device = "cuda" if torch.cuda.is_available() else "cpu"

    # Dataset paths
    img_dir = ["/content/drive/MyDrive/fastsam/FastSAM-main/test"]
    coco = "/content/drive/MyDrive/fastsam/FastSAM-main/test/_annotations.coco.json"

    validation_transform = A.Compose([
        A.Resize(1024, 1024),
        A.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
        ToTensorV2()
    ])

    # Carica dataset
    datasetTest = DatasetTest(image_dirs=img_dir, coco=coco, transform=validation_transform)
    dataloaderTest = DataLoader(datasetTest, batch_size=1, shuffle=False, collate_fn=collate_fn)

    save_dir = "results_yolo_dice"
    os.makedirs(save_dir, exist_ok=True)

    model = YOLOInferenceWrapper(
        model_path='/content/drive/MyDrive/yolo/segment/instrument_seg/weights/best.pt',
        device=device
    )

    all_preds_global = []
    all_gts_global = []
    per_image_metrics = []
    image_shape = (1024, 1024)
    iou_threshold = 0.5

    print("\n🔍 Inizio inferenza...")

    img_id = 0

    for images, labels in dataloaderTest:
        images_tensor = torch.stack(images).to(device)

        for idx in range(len(images)):
            image = images_tensor[idx]
            label = labels[idx]

            bboxes_gt = [[int(x), int(y), int(x+w), int(y+h)] for (x,y,w,h) in label]

            start_time = time.time()
            _, _, pred_boxes, pred_scores, masks = model.predict_from_tensor(
                image, conf=0.25, iou=0.5, imgsz=1024
            )
            latency = (time.time() - start_time) * 1000

            # Prepara predizioni
            bboxes_pred = []
            if pred_boxes is not None:
                for box, score in zip(pred_boxes, pred_scores):
                    bboxes_pred.append(list(box) + [float(score)])
                    all_preds_global.append(list(box) + [float(score), img_id])

            # Prepara GT
            for box in bboxes_gt:
                all_gts_global.append(box + [img_id])

            # Calcola metriche BOX-LEVEL + PIXEL-LEVEL
            metrics = calculate_metrics_per_image(bboxes_pred, bboxes_gt, 0.5, image_shape)

            # Estrai metriche
            tp = metrics['tp']
            fp = metrics['fp']
            fn = metrics['fn']

            # Box-level
            sensitivity_box = metrics['sensitivity_box']
            specificity_box = metrics['specificity_box']
            mean_dice_box = metrics['mean_dice_box']

            # Pixel-level
            tp_pixels = metrics['tp_pixels']
            fp_pixels = metrics['fp_pixels']
            fn_pixels = metrics['fn_pixels']
            tn_pixels = metrics['tn_pixels']
            sensitivity_pixel = metrics['sensitivity_pixel']
            specificity_pixel = metrics['specificity_pixel']
            precision_pixel = metrics['precision_pixel']
            dice_pixel = metrics['dice_pixel']
            iou_pixel = metrics['iou_pixel']
            accuracy_pixel = metrics['accuracy_pixel']

            # Box-level precision/recall/f1
            precision_box = tp / (tp + fp) if (tp + fp) > 0 else 0.0
            recall_box = tp / (tp + fn) if (tp + fn) > 0 else 0.0
            f1_box = 2 * precision_box * recall_box / (precision_box + recall_box) \
                if (precision_box + recall_box) > 0 else 0.0

            mean_iou_box = 0.0
            if metrics['matched_pairs']:
                ious = [calculate_iou_single(bboxes_pred[i], bboxes_gt[j])
                        for i, j, _ in metrics['matched_pairs']]
                mean_iou_box = np.mean(ious)

            # Salva metriche
            per_image_metrics.append({
                'img_id': img_id,
                'latency_ms': latency,
                'num_preds': len(bboxes_pred),
                'num_gts': len(bboxes_gt),
                'tp': tp,
                'fp': fp,
                'fn': fn,
                'precision_box': precision_box,
                'recall_box': recall_box,
                'f1_box': f1_box,
                'sensitivity_box': sensitivity_box,
                'specificity_box': specificity_box,
                'mean_iou_box': mean_iou_box,
                'mean_dice_box': mean_dice_box,
                'tp_pixels': tp_pixels,
                'fp_pixels': fp_pixels,
                'fn_pixels': fn_pixels,
                'tn_pixels': tn_pixels,
                'sensitivity_pixel': sensitivity_pixel,
                'specificity_pixel': specificity_pixel,
                'precision_pixel': precision_pixel,
                'dice_pixel': dice_pixel,
                'iou_pixel': iou_pixel,
                'accuracy_pixel': accuracy_pixel
            })

            # Visualizzazione ogni 10 immagini
            if img_id % 10 == 0:
                img_vis = image.detach().cpu().permute(1, 2, 0).numpy()
                img_vis = (img_vis - img_vis.min()) / (img_vis.max() - img_vis.min() + 1e-8)
                img_vis = (img_vis * 255).astype(np.uint8)
                img_vis = np.ascontiguousarray(img_vis)

                if img_vis.ndim == 2:
                    img_vis = cv2.cvtColor(img_vis, cv2.COLOR_GRAY2RGB)

                # Disegna GT boxes (verde)
                for box in bboxes_gt:
                    x1, y1, x2, y2 = map(int, box)
                    cv2.rectangle(img_vis, (x1, y1), (x2, y2), (0, 255, 0), 2)

                # Disegna predizioni (rosso)
                for bbox in bboxes_pred:
                    x1, y1, x2, y2 = map(int, bbox[:4])
                    cv2.rectangle(img_vis, (x1, y1), (x2, y2), (255, 0, 0), 2)

                # Crea visualizzazione
                fig, ax = plt.subplots(1, 1, figsize=(10, 10))
                ax.imshow(img_vis)
                title = (f"Img {img_id} | Box: D={mean_dice_box:.2f} S={sensitivity_box:.2f} | "
                        f"Pixel: D={dice_pixel:.2f} S={sensitivity_pixel:.2f}")
                ax.set_title(title, fontsize=11)
                ax.axis("off")

                save_path = os.path.join(save_dir, f"result_{img_id}.png")
                plt.savefig(save_path, bbox_inches="tight", dpi=100)
                plt.close(fig)

            if img_id % 10 == 0:
                print(f"[{img_id}] Box: Dice={mean_dice_box:.3f} | Pixel: Dice={dice_pixel:.3f} "
                      f"Sens={sensitivity_pixel:.3f} Spec={specificity_pixel:.3f}")

            img_id += 1

    # CALCOLA METRICHE GLOBALI
    print("\n" + "=" * 70)
    print("📊 METRICHE GLOBALI")
    print("=" * 70)

    df_metrics = pd.DataFrame(per_image_metrics)

    # Box-level globali
    total_tp = df_metrics['tp'].sum()
    total_fp = df_metrics['fp'].sum()
    total_fn = df_metrics['fn'].sum()

    global_precision_box = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0.0
    global_recall_box = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0.0
    global_f1_box = 2 * global_precision_box * global_recall_box / \
        (global_precision_box + global_recall_box) if (global_precision_box + global_recall_box) > 0 else 0.0

    # Pixel-level globali
    total_tp_pixels = df_metrics['tp_pixels'].sum()
    total_fp_pixels = df_metrics['fp_pixels'].sum()
    total_fn_pixels = df_metrics['fn_pixels'].sum()
    total_tn_pixels = df_metrics['tn_pixels'].sum()

    global_sensitivity_pixel = total_tp_pixels / (total_tp_pixels + total_fn_pixels) \
        if (total_tp_pixels + total_fn_pixels) > 0 else 0.0
    global_specificity_pixel = total_tn_pixels / (total_tn_pixels + total_fp_pixels) \
        if (total_tn_pixels + total_fp_pixels) > 0 else 0.0
    global_precision_pixel = total_tp_pixels / (total_tp_pixels + total_fp_pixels) \
        if (total_tp_pixels + total_fp_pixels) > 0 else 0.0
    global_dice_pixel = (2 * total_tp_pixels) / (2 * total_tp_pixels + total_fp_pixels + total_fn_pixels) \
        if (2 * total_tp_pixels + total_fp_pixels + total_fn_pixels) > 0 else 0.0
    global_iou_pixel = total_tp_pixels / (total_tp_pixels + total_fp_pixels + total_fn_pixels) \
        if (total_tp_pixels + total_fp_pixels + total_fn_pixels) > 0 else 0.0
    global_accuracy_pixel = (total_tp_pixels + total_tn_pixels) / \
        (total_tp_pixels + total_fp_pixels + total_fn_pixels + total_tn_pixels)

    # PR curve
    if len(all_preds_global) > 0:
        recalls, precisions, ap = compute_pr_curve_and_ap(
            all_preds_global, all_gts_global, iou_threshold
        )

        plt.figure(figsize=(8, 6))
        plt.plot(recalls, precisions, 'b-', linewidth=2, label=f'AP@{iou_threshold} = {ap:.4f}')
        plt.xlabel('Recall', fontsize=12)
        plt.ylabel('Precision', fontsize=12)
        plt.title('Precision-Recall Curve', fontsize=14)
        plt.legend(fontsize=11)
        plt.grid(True, alpha=0.3)
        plt.xlim([0, 1])
        plt.ylim([0, 1.05])
        plt.savefig(os.path.join(save_dir, 'pr_curve.png'), dpi=150, bbox_inches='tight')
        plt.close()
    else:
        ap = 0.0

    # STAMPA RISULTATI
    print(f"\n{'=' * 70}")
    print("📈 RISULTATI FINALI - BOX-LEVEL + PIXEL-LEVEL")
    print(f"{'=' * 70}\n")

    print(f"📊 Dataset:")
    print(f"   Totale immagini: {len(df_metrics)}")
    print(f"   Totale GT boxes: {len(all_gts_global)}")
    print(f"   Totale predizioni: {len(all_preds_global)}")

    print(f"\n🎯 METRICHE BOX-LEVEL (Globali):")
    print(f"   TP: {total_tp}, FP: {total_fp}, FN: {total_fn}")
    print(f"   Precision: {global_precision_box:.4f}")
    print(f"   Recall: {global_recall_box:.4f}")
    print(f"   F1-Score: {global_f1_box:.4f}")
    print(f"   AP@{iou_threshold}: {ap:.4f}")

    print(f"\n📊 METRICHE BOX-LEVEL (Medie per immagine):")
    print(f"   Precision:  {df_metrics['precision_box'].mean():.4f} ± {df_metrics['precision_box'].std():.4f}")
    print(f"   Recall:     {df_metrics['recall_box'].mean():.4f} ± {df_metrics['recall_box'].std():.4f}")
    print(f"   F1-Score:   {df_metrics['f1_box'].mean():.4f} ± {df_metrics['f1_box'].std():.4f}")
    print(f"   IoU medio:  {df_metrics['mean_iou_box'].mean():.4f} ± {df_metrics['mean_iou_box'].std():.4f}")
    print(f"   Dice medio: {df_metrics['mean_dice_box'].mean():.4f} ± {df_metrics['mean_dice_box'].std():.4f}")

    print(f"\n🎯 METRICHE PIXEL-LEVEL (Globali):")
    print(f"   TP pixels: {total_tp_pixels:,}, FP pixels: {total_fp_pixels:,}")
    print(f"   FN pixels: {total_fn_pixels:,}, TN pixels: {total_tn_pixels:,}")
    print(f"   Sensitivity: {global_sensitivity_pixel:.4f}")
    print(f"   Specificity: {global_specificity_pixel:.4f}")
    print(f"   Precision:   {global_precision_pixel:.4f}")
    print(f"   Dice Score:  {global_dice_pixel:.4f}")
    print(f"   IoU:         {global_iou_pixel:.4f}")
    print(f"   Accuracy:    {global_accuracy_pixel:.4f}")

    print(f"\n📊 METRICHE PIXEL-LEVEL (Medie per immagine):")
    print(f"   Sensitivity: {df_metrics['sensitivity_pixel'].mean():.4f} ± {df_metrics['sensitivity_pixel'].std():.4f}")
    print(f"   Specificity: {df_metrics['specificity_pixel'].mean():.4f} ± {df_metrics['specificity_pixel'].std():.4f}")
    print(f"   Precision:   {df_metrics['precision_pixel'].mean():.4f} ± {df_metrics['precision_pixel'].std():.4f}")
    print(f"   Dice Score:  {df_metrics['dice_pixel'].mean():.4f} ± {df_metrics['dice_pixel'].std():.4f}")
    print(f"   IoU:         {df_metrics['iou_pixel'].mean():.4f} ± {df_metrics['iou_pixel'].std():.4f}")
    print(f"   Accuracy:    {df_metrics['accuracy_pixel'].mean():.4f} ± {df_metrics['accuracy_pixel'].std():.4f}")

    print(f"\n⏱️  Performance:")
    print(f"   Latenza media: {df_metrics['latency_ms'].mean():.2f} ± {df_metrics['latency_ms'].std():.2f} ms")

    print(f"\n{'=' * 70}\n")

    # SALVA RISULTATI
    df_metrics.to_csv(os.path.join(save_dir, 'metrics_per_image.csv'), index=False)

    summary = {
        'total_images': len(df_metrics),
        'total_gt_boxes': len(all_gts_global),
        'total_predictions': len(all_preds_global),
        'global_precision': global_precision_box,
        'global_recall': global_recall_box,
        'global_sensitivity': global_sensitivity_pixel,
        'global_f1': global_f1_box,
        'ap_at_iou_threshold': ap,
        'iou_threshold': iou_threshold,
        'mean_precision': df_metrics['precision_box'].mean(),
        'mean_recall': df_metrics['recall_box'].mean(),
        'mean_sensitivity': df_metrics['sensitivity_box'].mean(),
        'mean_specificity': df_metrics['specificity_box'].mean(),
        'mean_f1': df_metrics['f1_box'].mean(),
        'mean_iou': df_metrics['mean_iou_box'].mean(),
        'mean_dice': df_metrics['mean_dice_box'].mean(),
        'mean_sensitivity_pixel': df_metrics['sensitivity_pixel'].mean(),
        'mean_specificity_pixel': df_metrics['specificity_pixel'].mean(),
        'mean_precision_pixel': df_metrics['precision_pixel'].mean(),
        'mean_dice_pixel': df_metrics['dice_pixel'].mean(),
        'mean_iou_pixel': df_metrics['iou_pixel'].mean(),
        'mean_latency_ms': df_metrics['latency_ms'].mean()
    }

    with open(os.path.join(save_dir, 'summary.txt'), 'w') as f:
        f.write("="*70 + "\n")
        f.write("YOLO EVALUATION - WITH DICE AND SENSITIVITY\n")
        f.write("="*70 + "\n\n")
        for key, value in summary.items():
            f.write(f"{key}: {value}\n")

    print(f"✅ Risultati salvati in '{save_dir}/'")
    print(f"   - metrics_per_image.csv: metriche dettagliate")
    print(f"   - pr_curve.png: curva Precision-Recall")
    print(f"   - summary.txt: riepilogo metriche globali")


if __name__ == "__main__":
    run_yolo_inference_with_dice()


🚀 YOLO INFERENCE - DICE + SENSITIVITY/SPECIFICITY
loading annotations into memory...
Done (t=0.01s)
creating index...
index created!
✓ Trovate 203 immagini con annotazioni

🔍 Inizio inferenza...
[0] Box: Dice=0.932 | Pixel: Dice=0.946 Sens=0.902 Spec=0.994
[10] Box: Dice=0.920 | Pixel: Dice=0.422 Sens=0.268 Spec=1.000
[20] Box: Dice=0.945 | Pixel: Dice=0.441 Sens=0.283 Spec=0.999
[30] Box: Dice=0.937 | Pixel: Dice=0.431 Sens=0.275 Spec=0.999
[40] Box: Dice=0.945 | Pixel: Dice=0.441 Sens=0.283 Spec=0.999
[50] Box: Dice=0.956 | Pixel: Dice=0.618 Sens=0.449 Spec=0.999
[60] Box: Dice=0.939 | Pixel: Dice=0.426 Sens=0.271 Spec=1.000
[70] Box: Dice=0.940 | Pixel: Dice=0.427 Sens=0.272 Spec=1.000
[80] Box: Dice=0.947 | Pixel: Dice=0.435 Sens=0.278 Spec=0.999
[90] Box: Dice=0.941 | Pixel: Dice=0.605 Sens=0.434 Spec=0.999
[100] Box: Dice=0.903 | Pixel: Dice=0.878 Sens=0.795 Spec=0.987
[110] Box: Dice=0.000 | Pixel: Dice=0.000 Sens=0.000 Spec=1.000
[120] Box: Dice=0.000 | Pixel: Dice=0.000 Sens=0